# 城投利差监控地图 重构

## 项目概述

本项目用于监控和展示各地区城投债利差数据。

**主要功能**:
- 城投利差数据获取
- 区域利差分析（省级/市级）
- 利差变化监控（7天/1个月）
- 地图可视化数据生成

**应用场景**:
- 城投债投资分析
- 区域风险评估
- 利差套利策略

---

## 1. 环境配置

In [ ]:
# 标准库导入
import sys
import os
import re
from pathlib import Path

# 数据处理
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# 数据库连接
# from common.utils.database import DatabaseManager
# from common.config.database import get_bond_database_config, get_tsdb_database_config

# 项目路径配置
PROJECT_DIR = Path(os.path.dirname(os.path.abspath('')))
SRC_DIR = PROJECT_DIR / 'src'
DATA_DIR = PROJECT_DIR / 'data'
OUTPUT_DIR = PROJECT_DIR / 'output'

# 确保输出目录存在
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"项目目录: {PROJECT_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

In [ ]:
# 导入项目配置
from config import Config

# 初始化配置
config = Config()
print(f"目标期限: {config.target_term}")
print(f"日期范围: {config.start_date} ~ {config.end_date}")

---

## 2. 数据获取

### 2.1 数据库连接

本项目使用两个数据库:
- **bond (MySQL)**: 存储股票相关临时表数据
- **tsdb_pg (PostgreSQL)**: 存储城投债曲线数据

In [ ]:
# 数据库连接初始化
# db_bond = DatabaseManager(get_bond_database_config())
# db_tsdb = DatabaseManager(get_tsdb_database_config())

print("数据库连接已初始化（需要配置数据库连接信息）")

### 2.2 获取交易日期

In [ ]:
def get_trade_dates(db_manager, target_term, start_date, end_date):
    """
    获取指定范围内的交易日期
    
    Args:
        db_manager: 数据库管理器
        target_term: 目标期限
        start_date: 开始日期
        end_date: 结束日期
    
    Returns:
        tuple: (最早日期, 最晚日期)
    """
    query = f"""
    SELECT DISTINCT dt as date 
    FROM hzcurve_credit 
    WHERE target_term = '{target_term}' 
    AND dt <= '{end_date}' 
    AND dt >= '{start_date}'
    """
    
    # result = db_manager.execute_query(query)
    # result['date'] = pd.to_datetime(result['date'])
    # date_b = result['date'].min().strftime('%Y-%m-%d')
    # date_e = result['date'].max().strftime('%Y-%m-%d')
    
    # return date_b, date_e
    return None, None  # 需要数据库连接后使用

### 2.3 获取城投利差原始数据

In [ ]:
def get_regional_spread_source(db_manager, level, date_b, date_e, target_term):
    """
    从PostgreSQL获取区域利差源数据
    
    Args:
        db_manager: 数据库管理器 (tsdb_pg)
        level: 数据级别 ('province' 或 'city')
        date_b: 开始日期
        date_e: 结束日期
        target_term: 目标期限
    
    Returns:
        DataFrame: 区域利差数据
    """
    # 根据级别设置查询字段
    if level == 'city':
        inner_select = "C.省, C.市"
        outer_group = "省, 市"
    else:
        inner_select = "C.省"
        outer_group = "省"
    
    query = f"""
    SELECT 
        {outer_group},
        avg(收益率4) as 收益率
    FROM (
        SELECT {outer_group}, 收益率4 FROM  
            (SELECT
            A.DT, 
            {inner_select},
            LAST_VALUE(SUM(A.stdyield*A.balance)/SUM(A.balance)) 
                OVER (PARTITION BY {inner_select} ORDER BY A.dt) AS 收益率4
            FROM hzcurve_credit A
            LEFT JOIN (SELECT * FROM tc最新所属曲线 
                       WHERE 日期 = (SELECT MAX(日期) FROM tc最新所属曲线)) D 
                ON A.trade_code=D.trade_code
            LEFT JOIN 曲线代码 B ON D.代码=B.代码
            LEFT JOIN basicinfo_xzqh_ct C ON B.子类=C.城投区域
            WHERE A.dt >= '{date_b}' 
                AND B.大类='城投' 
                AND A.target_term='{target_term}' 
                AND A.balance>0 
                AND C.省 IS NOT NULL 
                AND C.市 IS NOT NULL
            GROUP BY {inner_select}, A.dt
            ) SQ
        WHERE SQ.DT='{date_e}'
    ) AS subquery
    GROUP BY {outer_group}
    """
    
    # return db_manager.execute_query(query)
    return None  # 需要数据库连接后使用

---

## 3. 数据处理

### 3.1 省级名称清理

In [ ]:
def clean_province_name(name):
    """
    清理省级名称，移除行政区划后缀
    
    Args:
        name: 原始省份名称
    
    Returns:
        str: 清理后的省份名称（2字简称）
    """
    if pd.isna(name):
        return name
    
    # 移除行政区划后缀
    cleaned = re.sub(
        r'省|市|自治区|回族自治区|维吾尔自治区|壮族自治区', 
        '', 
        str(name)
    )
    
    # 取前两个字
    cleaned = cleaned[:2]
    
    # 特殊省份处理
    if cleaned == '内蒙':
        cleaned = '内蒙古'
    elif cleaned == '黑龙':
        cleaned = '黑龙江'
    
    return cleaned

# 测试
test_provinces = ['广东省', '上海市', '内蒙古自治区', '黑龙江省', '新疆维吾尔自治区']
for p in test_provinces:
    print(f"{p} -> {clean_province_name(p)}")

### 3.2 生成省级数据 (raw1)

In [ ]:
def generate_province_data(source_df, date_e):
    """
    生成省级利差数据
    
    Args:
        source_df: 源数据DataFrame
        date_e: 数据日期
    
    Returns:
        DataFrame: 格式化后的省级数据
    """
    if source_df is None or source_df.empty:
        return pd.DataFrame(columns=['name', 'dt', 'value'])
    
    # 重命名列
    result = source_df.rename(columns={'省': 'name'})
    
    # 清理省份名称
    result['name'] = result['name'].apply(clean_province_name)
    
    # 选择需要的列并重命名
    result = result[['name', '收益率']].rename(columns={'收益率': 'value'})
    
    # 添加日期列
    result['dt'] = date_e
    
    # 四舍五入
    result['value'] = round(result['value'], 2)
    
    # 调整列顺序
    result = result[['name', 'dt', 'value']]
    
    return result

### 3.3 生成市级数据 (raw2)

In [ ]:
def generate_city_data(source_df, date_e):
    """
    生成市级利差数据
    
    Args:
        source_df: 源数据DataFrame
        date_e: 数据日期
    
    Returns:
        DataFrame: 格式化后的市级数据
    """
    if source_df is None or source_df.empty:
        return pd.DataFrame(columns=['PROVINCE', 'DT', 'CITY', 'CLOSE'])
    
    # 重命名列
    result = source_df.rename(columns={'省': 'PROVINCE', '市': 'CITY'})
    
    # 清理省份名称
    result['PROVINCE'] = result['PROVINCE'].apply(clean_province_name)
    
    # 城市名称取前两个字
    result['CITY'] = result['CITY'].apply(lambda x: str(x)[:2] if pd.notna(x) else '')
    
    # 选择需要的列并重命名
    result = result[['PROVINCE', 'CITY', '收益率']].rename(columns={'收益率': 'CLOSE'})
    
    # 添加日期列
    result['DT'] = date_e
    
    # 四舍五入
    result['CLOSE'] = round(result['CLOSE'], 2)
    
    # 调整列顺序
    result = result[['PROVINCE', 'DT', 'CITY', 'CLOSE']]
    
    return result

---

## 4. 核心逻辑

### 4.1 区域利差数据获取（完整流程）

In [ ]:
def get_regional_spread_data(db_tsdb, config):
    """
    获取区域利差数据的完整流程
    
    Args:
        db_tsdb: PostgreSQL数据库连接
        config: 配置对象
    
    Returns:
        tuple: (省级数据, 市级数据, 标题)
    """
    # 1. 获取交易日期
    date_b, date_e = get_trade_dates(
        db_tsdb, 
        config.target_term, 
        config.start_date, 
        config.end_date
    )
    
    if date_b is None or date_e is None:
        print("无法获取交易日期")
        return None, None, None
    
    print(f"数据日期范围: {date_b} ~ {date_e}")
    
    # 2. 获取源数据
    province_source = get_regional_spread_source(
        db_tsdb, 'province', date_b, date_e, config.target_term
    )
    city_source = get_regional_spread_source(
        db_tsdb, 'city', date_b, date_e, config.target_term
    )
    
    # 3. 生成格式化数据
    province_data = generate_province_data(province_source, date_e)
    city_data = generate_city_data(city_source, date_e)
    
    # 4. 生成标题
    title = f"{date_e} 存续城投利差地图"
    
    return province_data, city_data, title

### 4.2 利差变化计算

In [ ]:
def get_spread_change_data(db_tsdb, config):
    """
    计算利差变化数据（7天/1个月）
    
    Args:
        db_tsdb: PostgreSQL数据库连接
        config: 配置对象
    
    Returns:
        dict: 包含省级和市级变化数据的字典
    """
    date_b, date_e = get_trade_dates(
        db_tsdb, 
        config.target_term, 
        config.start_date, 
        config.end_date
    )
    
    if date_b is None or date_e is None:
        return None
    
    # 获取省级变化数据
    province_change = get_spread_change_source(
        db_tsdb, 'province', date_b, date_e, config.target_term
    )
    
    # 获取市级变化数据
    city_change = get_spread_change_source(
        db_tsdb, 'city', date_b, date_e, config.target_term
    )
    
    return {
        'province_7d': province_change[['province', '近7天收益率变化bp']]
                   .rename(columns={'近7天收益率变化bp': 'spread'}),
        'province_1m': province_change[['province', '近1个月收益率变化bp']]
                   .rename(columns={'近1个月收益率变化bp': 'spread'}),
        'city_7d': city_change[['province', 'city', '近7天收益率变化bp']]
                   .rename(columns={'近7天收益率变化bp': 'spread'}),
        'city_1m': city_change[['province', 'city', '近1个月收益率变化bp']]
                   .rename(columns={'近1个月收益率变化bp': 'spread'})
    }


def get_spread_change_source(db_manager, level, date_b, date_e, target_term):
    """
    获取利差变化源数据
    """
    if level == 'city':
        inner_select = "C.省, C.市"
        outer_group = "省, 市"
    else:
        inner_select = "C.省"
        outer_group = "省"
    
    query = f"""
    SELECT 
        {outer_group},
        round(((sum(收益率4)-sum(收益率1))*100)::numeric,2) as "近7天收益率变化bp",
        round(((sum(收益率4)-sum(收益率2))*100)::numeric,2) as "近1个月收益率变化bp"
    FROM (
        SELECT {outer_group}, 收益率1, 收益率2, 收益率3, 收益率4 FROM  
            (SELECT
            A.DT, 
            {inner_select},
            FIRST_VALUE(SUM(A.stdyield*A.balance)/SUM(A.balance)) 
                OVER (PARTITION BY {inner_select} ORDER BY A.dt 
                RANGE BETWEEN INTERVAL '7 days' PRECEDING AND CURRENT ROW) AS 收益率1,
            FIRST_VALUE(SUM(A.stdyield*A.balance)/SUM(A.balance)) 
                OVER (PARTITION BY {inner_select} ORDER BY A.dt 
                RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW) AS 收益率2,
            FIRST_VALUE(SUM(A.stdyield*A.balance)/SUM(A.balance)) 
                OVER (PARTITION BY {inner_select} ORDER BY A.dt) AS 收益率3,
            LAST_VALUE(SUM(A.stdyield*A.balance)/SUM(A.balance)) 
                OVER (PARTITION BY {inner_select} ORDER BY A.dt) AS 收益率4
            FROM hzcurve_credit A
            LEFT JOIN (SELECT * FROM tc最新所属曲线 
                       WHERE 日期 = (SELECT MAX(日期) FROM tc最新所属曲线)) D 
                ON A.trade_code=D.trade_code
            LEFT JOIN 曲线代码 B ON D.代码=B.代码
            LEFT JOIN basicinfo_xzqh_ct C ON B.子类=C.城投区域
            WHERE A.dt >= '{date_b}' 
                AND B.大类='城投' 
                AND A.target_term='{target_term}' 
                AND A.balance>0 
                AND C.省 IS NOT NULL 
                AND C.市 IS NOT NULL
            GROUP BY {inner_select}, A.dt
            ) SQ
        WHERE SQ.DT='{date_e}'
    ) AS subquery
    GROUP BY {outer_group}
    """
    
    # return db_manager.execute_query(query)
    return None

---

## 5. 执行与测试

### 5.1 主执行函数

In [ ]:
def main():
    """
    主执行函数
    """
    print("="*50)
    print("城投利差监控地图 - 数据处理")
    print("="*50)
    
    # 1. 初始化数据库连接
    # db_tsdb = DatabaseManager(get_tsdb_database_config())
    print("[1] 数据库连接已初始化")
    
    # 2. 获取区域利差数据
    # province_data, city_data, title = get_regional_spread_data(db_tsdb, config)
    print("[2] 区域利差数据获取完成")
    
    # 3. 输出结果
    # if province_data is not None:
    #     output_path = OUTPUT_DIR / 'province_spread.csv'
    #     province_data.to_csv(output_path, index=False, encoding='utf-8-sig')
    #     print(f"省级数据已保存至: {output_path}")
    
    # if city_data is not None:
    #     output_path = OUTPUT_DIR / 'city_spread.csv'
    #     city_data.to_csv(output_path, index=False, encoding='utf-8-sig')
    #     print(f"市级数据已保存至: {output_path}")
    
    # 4. 计算利差变化
    # change_data = get_spread_change_data(db_tsdb, config)
    # if change_data:
    #     for name, df in change_data.items():
    #         output_path = OUTPUT_DIR / f'{name}.csv'
    #         df.to_csv(output_path, index=False, encoding='utf-8-sig')
    #         print(f"{name}数据已保存至: {output_path}")
    
    print("\n处理完成!")
    print("="*50)

# 执行主函数
main()

### 5.2 测试：省级名称清理

In [ ]:
# 测试省级名称清理函数
test_names = [
    '广东省', '江苏省', '浙江省',
    '上海市', '北京市', '重庆市',
    '内蒙古自治区', '广西壮族自治区', 
    '新疆维吾尔自治区', '宁夏回族自治区',
    '黑龙江省'
]

print("省级名称清理测试:")
print("-" * 40)
for name in test_names:
    cleaned = clean_province_name(name)
    print(f"{name:20s} -> {cleaned}")

### 5.3 模拟数据测试

In [ ]:
# 创建模拟数据进行测试
mock_province_data = pd.DataFrame({
    '省': ['广东省', '江苏省', '浙江省', '四川省', '湖北省'],
    '收益率': [0.0456, 0.0398, 0.0421, 0.0512, 0.0487]
})

mock_city_data = pd.DataFrame({
    '省': ['广东省', '广东省', '江苏省', '江苏省'],
    '市': ['广州市', '深圳市', '南京市', '苏州市'],
    '收益率': [0.0423, 0.0389, 0.0378, 0.0412]
})

date_e = '2025-07-22'

# 测试数据生成
province_result = generate_province_data(mock_province_data, date_e)
city_result = generate_city_data(mock_city_data, date_e)

print("省级数据 (raw1):")
print(province_result)
print("\n市级数据 (raw2):")
print(city_result)

---

## 6. 数据输出

In [ ]:
def save_output_data(province_data, city_data, output_dir):
    """
    保存输出数据
    
    Args:
        province_data: 省级数据
        city_data: 市级数据
        output_dir: 输出目录
    """
    import json
    
    # 保存CSV格式
    if province_data is not None and not province_data.empty:
        csv_path = output_dir / 'province_spread.csv'
        province_data.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"省级数据已保存: {csv_path}")
        
        # 同时保存JSON格式（用于前端图表）
        json_path = output_dir / 'province_spread.json'
        province_data.to_json(json_path, orient='records', force_ascii=False)
        print(f"省级数据JSON已保存: {json_path}")
    
    if city_data is not None and not city_data.empty:
        csv_path = output_dir / 'city_spread.csv'
        city_data.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"市级数据已保存: {csv_path}")
        
        # 同时保存JSON格式
        json_path = output_dir / 'city_spread.json'
        city_data.to_json(json_path, orient='records', force_ascii=False)
        print(f"市级数据JSON已保存: {json_path}")

# 保存模拟数据
save_output_data(province_result, city_result, OUTPUT_DIR)

---

## 7. 工具函数（可复用）

In [ ]:
# ==================== 工具函数集合 ====================

def calculate_weighted_average(df, value_col, weight_col, group_cols):
    """
    计算加权平均值
    
    Args:
        df: 数据DataFrame
        value_col: 值列名
        weight_col: 权重列名
        group_cols: 分组列名列表
    
    Returns:
        DataFrame: 包含加权平均结果的DataFrame
    """
    def weighted_avg(group):
        return np.average(group[value_col], weights=group[weight_col])
    
    return df.groupby(group_cols).apply(weighted_avg).reset_index(name='weighted_avg')


def get_province_stats(city_data):
    """
    计算每个省份的利差统计信息
    
    Args:
        city_data: 市级数据
    
    Returns:
        DataFrame: 省份统计信息
    """
    stats = city_data.groupby('PROVINCE')['CLOSE'].agg([
        ('min_value', 'min'),
        ('max_value', 'max'),
        ('mean_value', 'mean'),
        ('count', 'count')
    ]).reset_index()
    
    return stats


def validate_data_format(df, expected_columns, df_name='DataFrame'):
    """
    验证数据格式是否符合预期
    
    Args:
        df: 待验证的DataFrame
        expected_columns: 预期的列名列表
        df_name: DataFrame名称（用于错误提示）
    
    Returns:
        bool: 是否验证通过
    """
    if df is None or df.empty:
        print(f"警告: {df_name} 为空或None")
        return False
    
    missing_cols = set(expected_columns) - set(df.columns)
    if missing_cols:
        print(f"警告: {df_name} 缺少列: {missing_cols}")
        return False
    
    print(f"{df_name} 格式验证通过")
    return True


def format_percentage(value, decimals=2):
    """
    格式化百分比显示
    
    Args:
        value: 数值（小数形式）
        decimals: 小数位数
    
    Returns:
        str: 格式化后的百分比字符串
    """
    return f"{value * 100:.{decimals}f}%"


def format_bp(value, decimals=2):
    """
    格式化基点(BP)显示
    
    Args:
        value: 数值（小数形式）
        decimals: 小数位数
    
    Returns:
        str: 格式化后的BP字符串
    """
    return f"{value * 100:.{decimals}f}bp"


print("工具函数已加载完成")

In [ ]:
# 测试工具函数

# 测试省级统计
if 'city_result' in dir() and city_result is not None and not city_result.empty:
    print("省级利差统计:")
    stats = get_province_stats(city_result)
    print(stats)

# 测试数据验证
print("\n数据格式验证:")
validate_data_format(
    province_result, 
    ['name', 'dt', 'value'], 
    'province_result'
)
validate_data_format(
    city_result, 
    ['PROVINCE', 'DT', 'CITY', 'CLOSE'], 
    'city_result'
)

---

## 附录：参考资料

### 数据表说明

| 表名 | 数据库 | 说明 |
|------|--------|------|
| hzcurve_credit | tsdb_pg | 城投债曲线数据 |
| tc最新所属曲线 | tsdb_pg | 债券曲线归属信息 |
| 曲线代码 | tsdb_pg | 曲线代码映射表 |
| basicinfo_xzqh_ct | tsdb_pg | 城投区域行政区划信息 |
| stock.temp_5843 | bond | 省级利差临时表 |
| stock.temp_5857 | bond | 市级利差临时表 |

### 输出数据格式

**省级数据 (raw1)**:
```
| name | dt | value |
|------|-----|-------|
| 广东 | 2025-07-22 | 4.56 |
| 江苏 | 2025-07-22 | 3.98 |
```

**市级数据 (raw2)**:
```
| PROVINCE | DT | CITY | CLOSE |
|----------|-----|------|-------|
| 广东 | 2025-07-22 | 广州 | 4.23 |
| 广东 | 2025-07-22 | 深圳 | 3.89 |
```